# House Price Prediction with Multiple Linear Regression

**Module:** Machine Learning  |  **Class:** ETT Y4 BTech, Kigali College (Rwanda Polytechnic)

A real-estate agency in Rwanda wants a quick way to estimate a house's price (in million RWF) from its size, rooms, age, distance to the city centre, parking spaces and neighbourhood. In this notebook I clean the raw sales data, train a multiple linear regression model, evaluate it, and save it as `house_price_model.sav` so a Streamlit app can use it.

The model has the form

$$\hat{y} = b_0 + b_1x_1 + b_2x_2 + \dots + b_kx_k$$

where each $b_i$ tells us how much the price changes when feature $x_i$ goes up by one unit while the other features stay fixed.

## Part A: Data Exploration and Cleaning

### A1. Import libraries and load the data
Every library is imported once at the top, and each one has a job: `pandas` and `numpy` handle the data, `matplotlib` and `seaborn` draw charts, and `scikit-learn` builds and scores the model.

In [ ]:
import warnings
warnings.filterwarnings('ignore')            # keep the notebook output tidy

import numpy as np                           # numerical helpers (sqrt, arrays)
import pandas as pd                          # tables (DataFrames)
import matplotlib.pyplot as plt              # basic plotting
import seaborn as sns                        # nicer statistical plots
import joblib                                # save / load the trained model

from sklearn.model_selection import train_test_split          # 80/20 split
from sklearn.linear_model import LinearRegression             # the model
from sklearn.pipeline import Pipeline                         # chain steps together
from sklearn.compose import ColumnTransformer                 # different steps for different columns
from sklearn.impute import SimpleImputer                      # fill missing values
from sklearn.preprocessing import OneHotEncoder               # text categories -> 0/1 columns
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style='whitegrid')             # a clean chart style

In [ ]:
# Load the raw CSV into a DataFrame called df
df = pd.read_csv('house_price_prediction_dataset.csv')

print('Shape (rows, columns):', df.shape)    # how big is the data?
df.head(10)                                  # first 10 rows to eyeball the values

In [ ]:
df.info()        # column names, data types, and count of non-null values per column

In [ ]:
df.describe().T  # count, mean, std, min, quartiles, max for every numeric column (transposed to read easily)

**What I notice already:** the count column shows some variables have fewer than 125 values (so there are gaps), and the `max` values look suspicious: an `Area_m2` of 1200, a `Distance_to_City_km` of 85 and a price of 1200 million RWF are far bigger than the typical rows. I check these properly below.

### A2. Missing values
`isnull()` marks every empty cell as `True`, and `sum()` counts the `True`s per column.

In [ ]:
missing = df.isnull().sum()                  # number of blanks in each column
print(missing[missing > 0])                  # show only the columns that have blanks
print()
df[df.isnull().any(axis=1)]                  # show the actual rows that contain a blank

Nine rows each have exactly one blank. My decisions:

- **Missing target (`House_Price_Million_RWF`, 1 row): drop the row.** A row with no price cannot teach the model anything, and inventing a target value would just feed the model made-up answers.
- **Missing predictors (Area, Bedrooms, Bathrooms, Age, Distance, Parking, Neighborhood): impute.** Dropping 8 rows out of about 120 would waste data. I use the **median** for numbers (it is not pulled around by extreme values) and the **most frequent value** for `Neighborhood`. The imputation happens *inside the Pipeline* and is learned from the training data only, so the test set stays truly unseen and the saved model can also cope with blanks later.

In [ ]:
# Drop only the rows where the TARGET is missing
df = df.dropna(subset=['House_Price_Million_RWF'])
print('Rows left after dropping missing-price rows:', len(df))

### A3. Duplicate records
`duplicated()` flags a row if an identical row appeared earlier. `keep=False` in the first line shows *all* copies so we can see the pairs.

In [ ]:
print('Duplicate rows:', df.duplicated().sum())
df[df.duplicated(keep=False)].sort_values('House_ID')       # show originals and their copies together

In [ ]:
# Remove the repeated copies but keep the first occurrence of each house
df = df.drop_duplicates(keep='first').reset_index(drop=True)   # reset_index renumbers the rows 0,1,2...
print('Rows after removing duplicates:', len(df))
print('Is House_ID unique now?', df['House_ID'].is_unique)

The same three houses (IDs 1011, 1026, 1041) were entered more than once at the bottom of the file. Duplicates would count the same house several times and make the model look more accurate than it really is, so only the first copy is kept.

### A4. Outliers and impossible values
Two tools: **boxplots** (points beyond the whiskers are suspicious) and the **IQR rule**: anything below $Q1 - 1.5\times IQR$ or above $Q3 + 1.5\times IQR$ is flagged.

In [ ]:
num_cols = ['Area_m2', 'House_Age_Years', 'Distance_to_City_km', 'House_Price_Million_RWF']

fig, axes = plt.subplots(1, 4, figsize=(16, 4))          # 1 row, 4 charts side by side
for ax, col in zip(axes, num_cols):                      # loop: one boxplot per column
    sns.boxplot(y=df[col], ax=ax, color='#8fb8de')
    ax.set_title(col)
plt.tight_layout()
plt.savefig('fig_boxplots_before.png', dpi=120)
plt.show()

In [ ]:
def iqr_outliers(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)   # 25th and 75th percentiles
    iqr = q3 - q1                                           # spread of the middle 50%
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr              # the fences
    return low, high, series[(series < low) | (series > high)]

for col in ['Area_m2', 'Bedrooms', 'Bathrooms', 'House_Age_Years',
            'Distance_to_City_km', 'Parking_Spaces', 'House_Price_Million_RWF']:
    low, high, out = iqr_outliers(df[col])
    print(f'{col:26s} fences=({low:8.1f}, {high:8.1f})  flagged={len(out)}  max={df[col].max()}')

In [ ]:
# The most extreme rows, so I can judge them one by one
df[(df.Area_m2 > 400) | (df.House_Age_Years > 50) |
   (df.Distance_to_City_km > 40) | (df.House_Price_Million_RWF > 400)]

The IQR rule flags a lot of rows, but **flagged does not always mean wrong**. I judged each one:

| Value | Verdict | Action |
|---|---|---|
| `Area_m2` = 850 and 1200 | Impossible: a 3 to 4 bedroom house on 850 to 1200 m² of floor area, priced around 150 million RWF, looks like a decimal typo (85.0 and 120.0) | Set to blank, then impute the median inside the pipeline |
| `Distance_to_City_km` = 85 | Every other house is within 27 km, and 85 km is another town altogether | Set to blank, impute |
| `House_Price` = 950 and 1200 | About 6 times bigger than any other price, with normal-sized houses, so the target itself is wrong and I cannot know the true value | **Drop the rows** |
| `Area_m2` between 230 and 430, prices 320 or less | Big houses with matching high prices, so these are real | Keep |
| `House_Age_Years` between 25 and 50 | Old houses are normal in real life | Keep |
| `House_Age_Years` = 75 | Rare but possible, the only such value | Keep (mentioned as a limitation) |

In [ ]:
# 1) Impossible FEATURE values -> NaN (they get imputed later inside the pipeline)
df.loc[df['Area_m2'] > 600, 'Area_m2'] = np.nan
df.loc[df['Distance_to_City_km'] > 40, 'Distance_to_City_km'] = np.nan

# 2) Impossible TARGET values -> drop the whole row
df = df[df['House_Price_Million_RWF'] <= 400].reset_index(drop=True)

print('Final shape:', df.shape)
print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
# Save the cleaned dataset (this file goes to GitHub too)
df.to_csv('cleaned_house_price_dataset.csv', index=False)
df.describe().T

### A5. Visualisations

In [ ]:
plt.figure(figsize=(7, 4))
sns.histplot(df['House_Price_Million_RWF'], bins=20, kde=True, color='#3b7dbf')   # histogram + smooth curve
plt.title('Distribution of house prices'); plt.xlabel('Price (million RWF)')
plt.savefig('fig_price_dist.png', dpi=120, bbox_inches='tight'); plt.show()

*Interpretation: most houses sell between roughly 100 and 200 million RWF, with a longer tail of expensive houses on the right.*

In [ ]:
plt.figure(figsize=(7, 4))
sns.scatterplot(data=df, x='Area_m2', y='House_Price_Million_RWF', hue='Neighborhood')
plt.title('Area vs price'); plt.xlabel('Area (m²)'); plt.ylabel('Price (million RWF)')
plt.savefig('fig_area_price.png', dpi=120, bbox_inches='tight'); plt.show()

*Interpretation: bigger houses tend to cost more, but the points are spread out, so area alone does not decide the price.*

In [ ]:
plt.figure(figsize=(8, 4))
sns.boxplot(data=df, x='Neighborhood', y='House_Price_Million_RWF', palette='Blues')
plt.title('Price by neighbourhood'); plt.ylabel('Price (million RWF)'); plt.xticks(rotation=20)
plt.savefig('fig_price_neigh.png', dpi=120, bbox_inches='tight'); plt.show()

*Interpretation: Kigali City has the highest typical price, while Huye and Kicukiro sit lower, so location clearly matters.*

In [ ]:
plt.figure(figsize=(8, 6))
corr = df.drop(columns=['House_ID']).corr(numeric_only=True)      # correlation of numeric columns only
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation heatmap')
plt.savefig('fig_heatmap.png', dpi=120, bbox_inches='tight'); plt.show()

*Interpretation: Bedrooms and Bathrooms move together strongly, and both are the numeric features most linked to price.*

### A6. Cleaning decisions

| Problem found | Where | Action taken | Reason |
|---|---|---|---|
| Missing value | `House_Price_Million_RWF` (1 row) | Row dropped | A house with no price cannot be used for training |
| Missing values | Area, Bedrooms, Bathrooms, Age, Distance, Parking, Neighborhood (1 to 2 each) | Median (numbers) or most frequent (category) inside the Pipeline | Keeps the rows, and learning the fill value from the training set only avoids leakage |
| Duplicate records | Houses 1011, 1026, 1041 repeated at the end of the file (5 extra rows) | Extra copies removed | Repeats would inflate performance |
| Impossible area | `Area_m2` = 850 and 1200 | Set to blank, then imputed | Likely decimal typos, and the row's other values are fine |
| Impossible distance | `Distance_to_City_km` = 85 | Set to blank, then imputed | Far outside the range of every other house |
| Impossible price | `House_Price` = 950 and 1200 | Rows dropped | The target is wrong and the true value is unknown |
| Text category | `Neighborhood` | One-hot encoding, one level dropped | Regression needs numbers, and dropping one level avoids the dummy-variable trap |
| Identifier | `House_ID` | Excluded from predictors | It is just a label with no predictive meaning |

## Part B: Building the Multiple Linear Regression Model

### B1. Define X and y
`X` holds the predictors (every column except the ID and the target) and `y` is the target. Double brackets are not needed for `y` here since it is a single column (a Series).

In [ ]:
target = 'House_Price_Million_RWF'
X = df.drop(columns=['House_ID', target])       # all 7 predictor columns
y = df[target]                                  # the value we want to predict

numeric_features = ['Area_m2', 'Bedrooms', 'Bathrooms', 'House_Age_Years',
                    'Distance_to_City_km', 'Parking_Spaces']
categorical_features = ['Neighborhood']

print('X shape:', X.shape, '| y shape:', y.shape)
X.columns.tolist()

### B2 and B3. Train/test split (80/20, `random_state=42`)
The split happens **before** any learning so the test rows stay completely unseen. `random_state=42` makes the shuffle repeatable, so anyone running this notebook gets the same rows in each set.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,        # 20% of rows go to the test set
    random_state=42       # fixed seed = reproducible split
)
print('Training rows:', len(X_train), '| Testing rows:', len(X_test))

### B2 and B4. Preprocessing + model in one Pipeline
A `ColumnTransformer` treats the two column types differently:

- numbers: fill blanks with the **median**;
- `Neighborhood`: fill blanks with the **most frequent** value, then **one-hot encode** with `drop='first'` (one neighbourhood becomes the reference group, which avoids the dummy-variable trap).

The `Pipeline` then hands the prepared table to `LinearRegression`. Because everything is in one object, the saved `.sav` file accepts *raw* inputs like `'Gasabo'`.

In [ ]:
numeric_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))           # fill numeric blanks with the median
])

categorical_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),   # fill category blanks with the mode
    ('onehot', OneHotEncoder(drop='first', handle_unknown='error'))  # text -> 0/1 columns, drop one
])

preprocess = ColumnTransformer([
    ('num', numeric_pipe, numeric_features),
    ('cat', categorical_pipe, categorical_features)
])

model = Pipeline([
    ('preprocess', preprocess),
    ('regressor', LinearRegression())
])

model.fit(X_train, y_train)        # learns imputation values, encodings AND regression coefficients
print('Model trained.')

### B5. Intercept and coefficients

In [ ]:
feature_names = model.named_steps['preprocess'].get_feature_names_out()   # names after encoding
feature_names = [n.split('__')[1] for n in feature_names]                 # strip the 'num__' / 'cat__' prefixes

reg = model.named_steps['regressor']
print('Intercept:', round(reg.intercept_, 3))

coef_table = pd.DataFrame({'Variable': feature_names, 'Coefficient': reg.coef_.round(4)})
coef_table

The baseline neighbourhood (dropped by `drop='first'`) is **Gasabo**, so each `Neighborhood_...` coefficient is the price difference compared with a Gasabo house.

### B6. Multicollinearity check
Two ways: the correlation matrix (already seen), and the **Variance Inflation Factor (VIF)**. VIF near 1 means no problem, above 5 deserves attention, above 10 is severe.

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm

# VIF needs the fully numeric (imputed and encoded) training matrix, plus a constant column
X_num = model.named_steps['preprocess'].transform(X_train)
X_num = pd.DataFrame(X_num.toarray() if hasattr(X_num, 'toarray') else X_num, columns=feature_names)
X_const = sm.add_constant(X_num)

vif = pd.DataFrame({
    'Variable': X_num.columns,
    'VIF': [variance_inflation_factor(X_const.values, i + 1) for i in range(X_num.shape[1])]
}).round(2)
vif

In [ ]:
print('Correlation Bedrooms vs Bathrooms:',
      round(X_train['Bedrooms'].corr(X_train['Bathrooms']), 3))

## Part C: Evaluation and Interpretation

### C1. Metrics on training and test sets
- **R²**: share of price variation the model explains (1 is perfect).
- **Adjusted R²**: R² corrected for the number of predictors, $1-(1-R^2)\frac{n-1}{n-p-1}$.
- **MAE**: average size of the error, in million RWF.
- **RMSE**: like MAE but punishes big mistakes more.

A big gap between train and test scores would signal overfitting.

In [ ]:
def evaluate(name, y_true, y_pred, n_features):
    n = len(y_true)
    r2 = r2_score(y_true, y_pred)
    adj_r2 = 1 - (1 - r2) * (n - 1) / (n - n_features - 1)      # adjusted R-squared formula
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))          # sqrt of MSE
    return {'Set': name, 'R2': round(r2, 4), 'Adj R2': round(adj_r2, 4),
            'MAE': round(mae, 2), 'RMSE': round(rmse, 2)}

p = len(feature_names)                      # number of predictors AFTER encoding
train_pred = model.predict(X_train)
test_pred = model.predict(X_test)

results = pd.DataFrame([evaluate('Train', y_train, train_pred, p),
                        evaluate('Test',  y_test,  test_pred,  p)])
results

### C2. Actual vs predicted (test set)

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test, test_pred, alpha=0.8)
lims = [min(y_test.min(), test_pred.min()), max(y_test.max(), test_pred.max())]
plt.plot(lims, lims, 'r--', label='Perfect prediction')       # the y = x line
plt.xlabel('Actual price (million RWF)'); plt.ylabel('Predicted price (million RWF)')
plt.title('Actual vs predicted (test set)'); plt.legend()
plt.savefig('fig_actual_vs_pred.png', dpi=120, bbox_inches='tight'); plt.show()

### C3. Residual diagnostics

In [ ]:
import scipy.stats as stats
residuals = y_test - test_pred          # residual = actual - predicted

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
axes[0].scatter(test_pred, residuals); axes[0].axhline(0, color='r', ls='--')
axes[0].set_title('Residuals vs predicted'); axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Residual')
sns.histplot(residuals, kde=True, ax=axes[1]); axes[1].set_title('Histogram of residuals')
stats.probplot(residuals, dist='norm', plot=axes[2]); axes[2].set_title('Q-Q plot')
plt.tight_layout()
plt.savefig('fig_residuals.png', dpi=120, bbox_inches='tight'); plt.show()

In [ ]:
# Extra numeric checks that support the assumption discussion
print('Mean of residuals:', round(residuals.mean(), 3))
print('Shapiro-Wilk normality test p-value:', round(stats.shapiro(residuals)[1], 4))
print('Durbin-Watson (about 2 = no autocorrelation):',
      round(sm.stats.durbin_watson(y_train - train_pred), 3))

**Reading the numbers:** the mean residual is about 0, the Shapiro-Wilk p-value (about 0.44) is well above 0.05 so there is no evidence against normal residuals, and a Durbin-Watson value near 2 suggests the errors are not correlated with each other.

### C4. Interpreting three coefficients

In [ ]:
coef = dict(zip(feature_names, reg.coef_))
print(f"Area_m2: +1 m² changes the price by {coef['Area_m2']:.3f} million RWF")
print(f"Bathrooms: +1 bathroom changes the price by {coef['Bathrooms']:.3f} million RWF")
print(f"House_Age_Years: +1 year changes the price by {coef['House_Age_Years']:.3f} million RWF")
print(f"Neighborhood_Kigali City vs Gasabo: {coef['Neighborhood_Kigali City']:.3f} million RWF")

**Plain-language meaning (holding everything else constant):**

- **Area_m2 (about +0.52):** each extra square metre adds roughly 0.5 million RWF to the price.
- **Bathrooms (about +18.0):** each extra bathroom adds roughly 18 million RWF. This is large partly because Bedrooms and Bathrooms overlap, so the model splits their shared effect unevenly between them.
- **House_Age_Years (about -0.62):** each extra year of age lowers the price by roughly 0.6 million RWF.
- **Neighborhood_Kigali City (about +22.6):** a house in Kigali City costs roughly 22.6 million RWF more than an otherwise identical house in Gasabo, the reference group.

### C5. Are the regression assumptions reasonable?

| Assumption | Verdict | Evidence |
|---|---|---|
| Linearity | Reasonably met | Residuals vs predicted show no clear curve |
| Independence | Reasonably met | Durbin-Watson is about 2. The rows are separate sales, though I cannot rule out time or location effects |
| Constant variance | Roughly met | The spread looks slightly wider for expensive houses, but with only 24 test points it is hard to be sure |
| Normal residuals | Reasonably met | Q-Q points follow the line and Shapiro-Wilk p is about 0.44, with a mild right skew |
| No severe multicollinearity | Partly met | Bedrooms and Bathrooms have correlation about 0.93 and VIF about 7.9, which is moderate to high. All other VIFs are below 2.2 |

**Performance:** the test R² is about 0.82 with an RMSE of about 23 million RWF, close to the training scores (R² about 0.86), so overfitting is mild. The adjusted R² on the test set (about 0.65) looks much lower only because the formula punishes 11 predictors on just 24 rows, so it should be read with care.

**Two limitations:**
1. The dataset is tiny (117 rows, with only 24 in the test set), so the metrics can change a lot with a different split, and the Bedrooms and Bathrooms coefficients are unstable because those two features move together.
2. A straight-line model cannot capture interactions or diminishing returns (for example, a large house in Kigali City may gain more per extra square metre than one elsewhere), and it should not be trusted outside the training ranges (for example, above 430 m² or for houses older than 50 years).

## Part D: Saving the Model as a `.sav` File

In [ ]:
joblib.dump(model, 'house_price_model.sav')       # writes the WHOLE pipeline to disk
print('Saved house_price_model.sav')

In [ ]:
# Fresh load, then prove the loaded copy behaves exactly like the original
loaded_model = joblib.load('house_price_model.sav')
same = np.allclose(model.predict(X_test), loaded_model.predict(X_test))
print('Predictions identical:', same)

In [ ]:
# Predict a brand new house of my own choice (column names must match training exactly)
new_house = pd.DataFrame([{
    'Area_m2': 120.0, 'Bedrooms': 3, 'Bathrooms': 2, 'House_Age_Years': 5.0,
    'Distance_to_City_km': 4.0, 'Parking_Spaces': 1, 'Neighborhood': 'Kicukiro'
}])
price = loaded_model.predict(new_house)[0]
print(f'Predicted price: {price:.1f} million RWF')

## Ranges for the Streamlit app
These come from the cleaned data and feed the sliders in `app.py`.

In [ ]:
df[numeric_features].agg(['min', 'median', 'max']).T